In [81]:
import pandas as pd

In [97]:
df = pd.read_csv("./IMDB Dataset.csv")

In [98]:
df.shape

(50000, 2)

In [99]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [100]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [101]:
df.drop_duplicates(inplace=True)

In [102]:
df.shape

(49582, 2)

## pre-processing

In [103]:
### 1. converting to lowercase
df["review"] = df["review"].str.lower()
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [104]:
### 2.Removing URLs
import re
def remove_urls(text):
    text = re.sub(r"http\S+","",text) # (pattern,repl,string)
    return text
df["review"] = df["review"].apply(remove_urls)
df.head(10)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
5,"probably my all-time favorite movie, a story o...",positive
6,i sure would like to see a resurrection of a u...,positive
7,"this show was an amazing, fresh & innovative i...",negative
8,encouraged by the positive comments about this...,negative
9,if you like original gut wrenching laughter yo...,positive


In [105]:
### Removing punctuations
def remove_punct(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
df["review"] = df["review"].apply(remove_punct)
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [106]:
### 4.Removing html tags
def remove_html(text):
    text = re.sub(r"<.*?>","",text) 
    return text
df["review"] = df["review"].apply(remove_html)
df.head(10)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive
5,probably my alltime favorite movie a story of ...,positive
6,i sure would like to see a resurrection of a u...,positive
7,this show was an amazing fresh innovative ide...,negative
8,encouraged by the positive comments about this...,negative
9,if you like original gut wrenching laughter yo...,positive


In [107]:
### removing the stopwords
#Natural lang tool kit
import nltk
nltk.download("punkt") ## tokenizer
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\eppah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\eppah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\eppah\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [108]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [43]:
# sample_text = "I like coding in python!"
# tokens = word_tokenize(sample_text)
# tokens

In [109]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")
    return text
df["review"] = df["review"].apply(remove_stopwords)  

In [110]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [111]:
### stemming -> running - run,played -> play
# we use PorterStemming-(popular stemming algo)
from nltk.stem import PorterStemmer

In [113]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words =[]

    tokens = word_tokenize(text)
    for word in tokens:
        stemmed_token = ps.stem(word)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"] = df["review"].apply(stemming)   

In [114]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [115]:
 ## Encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [117]:
y=df["sentiment"]

In [119]:
## vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])

In [120]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057140 stored elements and shape (49582, 5000)>

In [122]:
## Dataset and Data Loaders
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [131]:
import torch
from torch.utils.data import TensorDataset,DataLoader
X_test = X_test.toarray()

In [132]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [143]:
trainloader = DataLoader(train_set,shuffle=True,batch_size = 64)
testloader = DataLoader(test_set,shuffle=True,batch_size = 64)

### Build our RNN. 

In [144]:
import torch.nn as nn
import torch.optim as optim

In [145]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        ## RNN Layer
        self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        ## fully connected layer
        self.fc = nn.Linear(hidden_size,1)
    def forward(self,x):
        #optional
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out, _=self.rnn(x,h0)
        #1st val = hidden state of all the timestamps => (batch,seq_len,hidden_size)
        # 2nd va; = final hidden state of last timestamp
        out = self.fc(out[:,-1,:])

        return out

In [146]:
input_size = X_train.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [149]:
## Training the RNN
epochs = 10
for epoch in range(epochs):
    model.train()

    for xb,yb in trainloader:
        optimizer.zero_grad()

        xb=xb.unsqueeze(1) # add singleton direction
        outputs = model(xb)
        outputs = torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs,yb)
        loss.backward()
        optimizer.step()
    print(f"{epoch} and loss = {loss.item()}") 

0 and loss = 0.2045416682958603
1 and loss = 0.15354794263839722
2 and loss = 0.19516733288764954
3 and loss = 0.15131516754627228
4 and loss = 0.2002238631248474
5 and loss = 0.30446290969848633
6 and loss = 0.19142337143421173
7 and loss = 0.23154237866401672
8 and loss = 0.2206949144601822
9 and loss = 0.3185661733150482


In [150]:
model.eval()
with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for xb,yb in testloader:
        xb = xb.unsqueeze(1)

        outputs = model(xb)
        pred = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        tot_vals +=yb.size(0)
        correct_vals+=(pred==yb).sum().item()
    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.51981446001815
